# Case Study 2 — full pipeline (run top to bottom)

This one notebook runs everything on the university server: build the corpus, embed and index, generate, judge, score, and show the four-bucket result.

**Two switches, set in section 2:**
- **Generator** — the free dev model (Gemini) to shake out bugs now, or the frozen `claude-sonnet-4-6` for the real graded run.
- **Judge** — `stub` (offline, instant) for dev, or the frozen 70B open model via vLLM on this GPU for the real run.

Free-model runs are for **debugging the pipeline, not results**. The numbers that go in the manuscript use the frozen generator + the 70B judge.

Run the cells in order. Sections 3 and 4 build the retrieval store (once). Section 6 generates, section 7 judges and scores, section 8 shows the result.

In [1]:
!pkill -f vllm ; sleep 5 ; nvidia-smi  # kill orpahn server

## 0. Environment probe
Tells us what this server can do. Run it first.

In [1]:
import sys, os, subprocess, platform, urllib.request
print("python:", sys.version.split()[0], "|", platform.platform())
print("cwd:", os.getcwd())
if not os.path.exists("src/run_generation.py"):
    print("!! Run this notebook from the repo ROOT (the folder with src/, config/, test_set.jsonl).")

def check_internet(url="https://pypi.org", timeout=5):
    try:
        urllib.request.urlopen(url, timeout=timeout); return True
    except Exception as e:
        print("  internet check failed:", e); return False

HAS_INTERNET = check_internet()
print("internet:", HAS_INTERNET)

HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        p = torch.cuda.get_device_properties(0)
        print("GPU:", torch.cuda.get_device_name(0),
              f"| VRAM {p.total_memory/1e9:.0f} GB | count {torch.cuda.device_count()}")
    else:
        print("GPU: torch present but no CUDA device visible")
except Exception as e:
    print("GPU: torch not importable yet (install deps in section 1) ->", e)
print(f"\nSUMMARY  internet={HAS_INTERNET}  gpu={HAS_GPU}")

python: 3.11.13 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
cwd: /home/jovyan/case_study2
internet: True
GPU: NVIDIA RTX A6000 | VRAM 51 GB | count 2

SUMMARY  internet=True  gpu=True


## 1. Install dependencies (run once, needs internet)
vLLM for the 70B judge is heavy and installed later, only when you switch the judge on.

In [2]:
if HAS_INTERNET:
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=False)
    subprocess.run([sys.executable,"-m","pip","install","-q","openai"], check=False)
    print("core deps installed")
else:
    print("No internet here: install where there is internet, or pre-stage wheels.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.7.2 requires click~=8.1.7, but you have click 8.5.0 which is incompatible.
crewai 1.7.2 requires regex~=2024.9.11, but you have regex 2026.9.10 which is incompatible.
crewai 1.7.2 requires tokenizers~=0.20.3, but you have tokenizers 0.23.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


core deps installed



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


## 2. Config — the only knobs

For the **free dev run** (default): Gemini generator + stub judge. Paste your free Google AI Studio key below.

For the **real graded run**: set `GEN_PROVIDER="anthropic"`, `GEN_MODEL="claude-sonnet-4-6"`, paste `ANTHROPIC_API_KEY`, set `JUDGE="vllm"`, and `RUN_FULL=True`.

In [ ]:
# ---------- GENERATOR ----------
GEN_PROVIDER = "openai_compatible"      # "anthropic" for the frozen graded run
GEN_MODEL    = "openai/gpt-oss-120b"       # "claude-sonnet-4-6" for the frozen run
GEN_BASE_URL = "https://api.groq.com/openai/v1"
GEN_KEY_ENV  = "GROQ_API_KEY"
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "")   # <-- paste free key
# os.environ["ANTHROPIC_API_KEY"] = ""   # <-- paste for the frozen run

# ---------- JUDGE ----------
JUDGE          = "vllm"                  # "vllm" for the real 70B judge on this GPU
JUDGE_MODEL    = "Qwen/Qwen2.5-72B-Instruct-AWQ"   # or meta-llama/Llama-3.3-70B-Instruct
JUDGE_BASE_URL = "http://localhost:8000/v1"

# ---------- RUN SIZE ----------
RUN_FULL = False    # False = 8-row sanity per config; True = full 215 x 3

def sh(cmd):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    r = subprocess.run(cmd, shell=not isinstance(cmd, list), capture_output=True, text=True)
    if r.stdout: print(r.stdout[-6000:])
    if r.returncode != 0 and r.stderr: print("STDERR:\n", r.stderr[-4000:])
    return r.returncode

print("generator:", GEN_PROVIDER, GEN_MODEL, "| judge:", JUDGE, "| full run:", RUN_FULL)

generator: openai_compatible openai/gpt-oss-120b | judge: vllm | full run: False


## 3. Corpus (400 chunks from the frozen guidelines)
Uses the committed corpus if present; only rebuilds if missing (rebuild needs poppler/pdftotext).

In [5]:
CHUNKS = "results/corpus_chunks.jsonl"
PDF = "data/guidelines/Draft_Guidelines_on_the_classification_of_high_risk_AI_Annex_III.pdf"
if os.path.exists(CHUNKS):
    print("corpus present (committed):", sum(1 for _ in open(CHUNKS)), "chunks — skip rebuild")
else:
    sh([sys.executable, "src/build_corpus.py", PDF, CHUNKS])
    print("chunks:", sum(1 for _ in open(CHUNKS)))

corpus present (committed): 400 chunks — skip rebuild


## 4. Embed + index (downloads bge model, needs internet, builds Chroma)
The vector store is not in git, so build it here once.

In [6]:
sh([sys.executable, "src/embed_and_index.py", "config/pipeline.yaml"])

$ /usr/bin/python3 src/embed_and_index.py config/pipeline.yaml
loaded 400 chunks
embedded -> (400, 768)
persisted 400 vectors -> results/chroma/eu_ai_act_guidelines



0

## 5. Retrieval quality check (optional, no API needed)
Should show Hit@5 around 0.83.

In [7]:
sh([sys.executable, "src/retrieval_eval.py", "config/pipeline.yaml"])

$ /usr/bin/python3 src/retrieval_eval.py config/pipeline.yaml

Retrieval quality over 215 rows
  Hit@5   0.833   (PRIMARY)
  Hit@10  0.898
  Recall@5  0.246   Recall@10 0.374
  MRR     0.633
  wrote results/retrieval_eval.json and results/retrieval_eval.md



0

In [8]:
# patch: ride out the free-tier 5-requests-per-minute limit
import pathlib
p = pathlib.Path("src/run_generation.py")
s = p.read_text()
s = s.replace("def call_generator(client, kind, gen_cfg, system, user, max_retries=4):",
              "def call_generator(client, kind, gen_cfg, system, user, max_retries=8):")
s = s.replace("            time.sleep(2 ** attempt)",
              "            time.sleep(15)")
p.write_text(s)
print("patched: 8 retries, 15s wait on rate-limit")

patched: 8 retries, 15s wait on rate-limit


## 6. Generation
Writes one file per config to results/runs/. Sanity (8 rows) unless RUN_FULL=True.

In [9]:
gen_args = [sys.executable, "src/run_generation.py",
            "--provider", GEN_PROVIDER, "--model", GEN_MODEL,
            "--base-url", GEN_BASE_URL, "--api-key-env", GEN_KEY_ENV]

gen_args += ["--configs", "agent_structured"]
if not RUN_FULL:
    gen_args += ["--limit", "40"]
sh(gen_args)

import glob, json
for f in sorted(glob.glob("results/runs/*.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    print(os.path.basename(f), "->", len(rows), "rows | first: pred=%s gold=%s" %
          (rows[0]["pred_label"], rows[0]["gold_label"]))

$ /usr/bin/python3 src/run_generation.py --provider openai_compatible --model openai/gpt-oss-120b --base-url https://api.groq.com/openai/v1 --api-key-env GROQ_API_KEY --configs agent_structured --limit 40

=== agent_structured  (40 rows) -> results/runs/agent_structured.jsonl ===
  [1/40] anx3-001: pred=None gold=high-risk PARSE?
  [2/40] anx3-002: pred=high-risk gold=high-risk ok
  [3/40] anx3-003: pred=high-risk gold=high-risk ok
  [4/40] anx3-004: pred=high-risk gold=high-risk ok
  [5/40] anx3-005: pred=high-risk gold=high-risk ok
  [6/40] anx3-006: pred=high-risk gold=high-risk ok
  [7/40] anx3-007: pred=high-risk gold=high-risk ok
  [8/40] anx3-008: pred=not-high-risk gold=not-high-risk ok
  [9/40] anx3-009: pred=not-high-risk gold=not-high-risk ok
  [10/40] anx3-010: pred=not-high-risk gold=not-high-risk ok
  [11/40] anx3-011: pred=not-high-risk gold=not-high-risk PARSE?
  [12/40] anx3-012: pred=not-high-risk gold=not-high-risk ok
  [13/40] anx3-013: pred=high-risk gold=not-high-

## 7. Judge + scoring
`stub` = offline and instant (dev). `vllm` = the real 70B judge on this GPU: the next cell starts a vLLM server (first run downloads the 70B, can take a while), scores against it, then stops it.

In [10]:
import time, urllib.request, os
vllm_env = {**os.environ, "VLLM_USE_FLASHINFER_SAMPLER": "0"}
vllm_proc = None
if JUDGE == "vllm":
    if not HAS_GPU:
        print("JUDGE=vllm but no GPU detected.")
    else:
        subprocess.run([sys.executable,"-m","pip","install","-q","vllm"], check=False)
        vllm_proc = subprocess.Popen([sys.executable,"-m","vllm.entrypoints.openai.api_server",
            "--model", JUDGE_MODEL, "--port", "8000",
            "--dtype", "auto",
            "--gpu-memory-utilization", "0.95",
            "--max-model-len", "6144"],
            env=vllm_env)                      # <-- disables the FlashInfer sampler JIT
        print("starting vLLM (single GPU, no flashinfer sampler)...")
        up = False
        for _ in range(300):
            try:
                urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3)
                up = True; print("vLLM server is up"); break
            except Exception:
                time.sleep(10)
        if not up:
            print("vLLM did NOT come up — scroll up for the real error.")
else:
    print("JUDGE=stub — scoring runs offline and instant.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogen-core 0.7.5 requires protobuf~=5.29.3, but you have protobuf 6.33.6 which is incompatible.
crewai 1.7.2 requires click~=8.1.7, but you have click 8.5.0 which is incompatible.
crewai 1.7.2 requires openai~=1.83.0, but you have openai 3.13.0 which is incompatible.
crewai 1.7.2 requires opentelemetry-api~=1.34.0, but you have opentelemetry-api 1.44.0 which is incompatible.
crewai 1.7.2 requires opentelemetry-exporter-otlp-proto-http~=1.34.0, but you have opentelemetry-exporter-otlp-proto-http 1.44.0 which is incompatible.
crewai 1.7.2 requires opentelemetry-sdk~=1.34.0, but you have opentelemetry-sdk 1.44.0 which is incompatible.
crewai 1.7.2 requires pydantic~=2.11.9, but you have pydantic 2.13.5 which is incompatible.
crewai 1.7.2 requires regex~=2024.9.11, but you have regex 2026.9.10 which is incompatible.

starting vLLM (single GPU, no flashinfer sampler)...


/home/jovyan/.local/lib/python3.11/site-packages/vllm/entrypoints/openai/api_server.py:24: DeprecationWarning: `vllm.entrypoints.openai.api_server` is deprecated and will likely beunsupported in a future version. Use the corresponding function from `vllm.entrypoints.launchers` instead.
  warnings.warn(
/home/jovyan/.local/lib/python3.11/site-packages/vllm/entrypoints/openai/api_server.py:51: DeprecationWarning: The `python -m vllm.entrypoints.openai.api_server` command is deprecated and may be removed in a future release. Please use `vllm server` instead.
  warnings.warn(


(APIServer pid=2470) INFO 09-13 16:00:17 [api_utils.py:347] 
(APIServer pid=2470) INFO 09-13 16:00:17 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=2470) INFO 09-13 16:00:17 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=2470) INFO 09-13 16:00:17 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-72B-Instruct-AWQ
(APIServer pid=2470) INFO 09-13 16:00:17 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=2470) INFO 09-13 16:00:17 [api_utils.py:347] 
(APIServer pid=2470) INFO 09-13 16:00:17 [api_utils.py:286] non-default args: {'model': 'Qwen/Qwen2.5-72B-Instruct-AWQ', 'max_model_len': 6144, 'gpu_memory_utilization': 0.95}


(APIServer pid=2470) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


...

(EngineCore pid=3143) INFO 09-13 16:07:45 [model_runner.py:960] Graph capturing finished in 22 secs, took 0.85 GiB
(EngineCore pid=3143) INFO 09-13 16:07:46 [gpu_worker.py:625] Available KV cache memory: 2.66 GiB
(EngineCore pid=3143) INFO 09-13 16:07:46 [gpu_worker.py:640] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9500 is equivalent to --gpu-memory-utilization=0.9301 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9699. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=3143) INFO 09-13 16:07:46 [kv_cache_utils.py:2032] GPU KV cache size: 8,720 tokens, Maximum concurrency for 6,144 tokens per request: 1.42x
(EngineCore pid=3143) INFO 09-13 16:07:46 [kernel_warmup.py:124] JIT kernel warmup starting.
(EngineCore pid=3143) INFO 09-13 16:07:46 [kernel_warmup.py:134] JIT kernel warmup finished in 0.00s.


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:07<00:00,  4.75it/s]


(EngineCore pid=3143) INFO 09-13 16:08:16 [model_runner.py:960] Graph capturing finished in 29 secs, took 0.46 GiB
(EngineCore pid=3143) INFO 09-13 16:08:16 [gpu_worker.py:797] CUDA graph pool memory: 0.46 GiB (actual), 0.94 GiB (estimated), difference: 0.48 GiB (103.8%).
(EngineCore pid=3143) INFO 09-13 16:08:16 [gpu_worker.py:860] Free memory on device (47.14/47.4 GiB) on startup. Desired GPU memory utilization is (0.95, 45.03 GiB). Actual usage is 39.05 GiB for consumed memory (weights + non-torch), 3.32 GiB for peak activation, and 0.46 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=2206793012` (2.06 GiB) to fit into requested memory, or `--kv-cache-memory=4466641408` (4.16 GiB) to fully utilize gpu memory. Current kv cache memory in use is 2.66 GiB.
(EngineCore pid=3143) INFO 09-13 16:08:20 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=3143) INFO 09-13 16:08

(APIServer pid=2470) INFO:     Started server process [2470]
(APIServer pid=2470) INFO:     Waiting for application startup.
(APIServer pid=2470) INFO:     Application startup complete.


(APIServer pid=2470) INFO:     127.0.0.1:55876 - "GET /v1/models HTTP/1.1" 200 OK
vLLM server is up


In [11]:
score_args = [sys.executable, "src/run_scoring.py", "--judge", JUDGE]
if JUDGE == "vllm":
    score_args += ["--judge-model", JUDGE_MODEL, "--judge-base-url", JUDGE_BASE_URL]
sh(score_args)

if vllm_proc is not None:
    vllm_proc.terminate(); print("vLLM server stopped")

$ /usr/bin/python3 src/run_scoring.py --judge vllm --judge-model Qwen/Qwen2.5-72B-Instruct-AWQ --judge-base-url http://localhost:8000/v1
(APIServer pid=2470) INFO 09-13 16:08:34 [loggers.py:310] Engine 000: Avg prompt throughput: 28.5 tokens/s, Avg generation throughput: 3.5 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 4.4%, Prefix cache hit rate: 0.0%
(APIServer pid=2470) INFO:     127.0.0.1:55882 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=2470) INFO 09-13 16:08:44 [loggers.py:310] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 14.0 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 16.7%, Prefix cache hit rate: 0.0%
(APIServer pid=2470) INFO 09-13 16:08:54 [loggers.py:310] Engine 000: Avg prompt throughput: 145.2 tokens/s, Avg generation throughput: 16.1 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 18.6%, Prefix cache hit rate: 0.0%
(APIServer pid=2470) INFO 09-13 16:09:04 [loggers.py:31

## 8. Results — the four-bucket matrix
Correctness, faithfulness, then the right/wrong x faithful/unfaithful buckets. The off-diagonal rows are in results/scoring/buckets/*_offdiagonal.jsonl.

In [12]:
for f in ["results/scoring/correctness_summary.md",
          "results/scoring/faithfulness_summary.md",
          "results/scoring/buckets_summary.md"]:
    print("="*72); print(f); print("="*72)
    print(open(f).read() if os.path.exists(f) else "(not produced yet)")
    print()

results/scoring/correctness_summary.md
# Correctness (predicted label vs frozen Commission ground truth)

Positive class = high-risk. Accuracy plus per-class precision/recall/F1 because the set is imbalanced (frozen). Parse failures counted as incorrect and also shown separately.

| Config | n | Accuracy | HR precision | HR recall | HR F1 | Macro F1 | Parse fails |
|---|---|---|---|---|---|---|---|
| agent_structured | 40 | 0.775 | 0.773 | 0.850 | 0.810 | 0.799 | 3 |

## agent_structured

Confusion (high-risk positive): tp=17 fp=5 fn=3 tn=15

By edge-case type: none n=40 acc=0.775

By area: biometrics 0.812, critical-infrastructure 0.625


results/scoring/faithfulness_summary.md
# RAGAS faithfulness (explanation grounded in retrieved passages)

Judge: `openai-compatible:Qwen/Qwen2.5-72B-Instruct-AWQ`. Faithfulness = supported statements / total statements. Baseline1 has no retrieval, so no score by design. Faithfulness says nothing about label correctness -- that gap is the study's sub

(APIServer pid=2470) INFO:     Shutting down
(APIServer pid=2470) INFO:     Waiting for application shutdown.
(APIServer pid=2470) INFO:     Application shutdown complete.
/usr/lib/python3.11/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 1 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


In [35]:
nvidia-smi

NameError: name 'nvidia' is not defined

In [ ]:
curl -s http://localhost:8000/v1/models